# Analysis of theta-gamma comodulogram metrics (PAC amplitude, oscillatory event counts, theta-gamma modulation index)


In [69]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Binomial
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr 
from matplotlib.lines import Line2D

from figure_style import apply_style, set_panel_title, WT, NLGF
GENO_COLORS = {"NLGF": NLGF, "WT": WT}
GENO_ORDER = ["WT", "NLGF"]
apply_style()

In [70]:
# ─── Load data ────────────────────────────────────────────────────────────────
basepath = Path('/Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis')
LFP_CSV_PATH = basepath / 'concatenated_lfp_stats.csv' 
OUTPUT_DIR = LFP_CSV_PATH.parent.parent / 'LFP_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)


df = pd.read_csv(LFP_CSV_PATH)

# Clean names / categorical order
df["Genotype"] = pd.Categorical(df["Genotype"], categories=GENO_ORDER, ordered=True)
df["environment"] = df["environment"].astype(str)
df["mouse_name"] = df["mouse_name"].astype(str)
df["Experimenter"] = df["Experimenter"].astype(str)


## Step 1 — Normality (Shapiro–Wilk) and Step 2 — Levene's test, per metric × environment

In [71]:
# ─── Metrics to analyse ───────────────────────────────────────────────────────
metrics = {
    "comodulogram_PAC_slow_gamma": "Theta–slow gamma PAC",
    "comodulogram_PAC_fast_gamma": "Theta–fast gamma PAC",
    "n_oscillatory_epochs_slow_gamm": "Slow gamma event count",
    "n_oscillatory_epochs_fast_gamma": "Fast gamma event count",
    "theta_gamma_modulation_index_fast_gamma": "Theta–Fast gamma modulation index",
    "theta_gamma_modulation_index_slow_gamma": "Theta–Slow gamma modulation index",
}

metrics = {k: v for k, v in metrics.items() if k in df.columns}

# ─── Step 1: Normality (Shapiro-Wilk) + Step 2: Levene's test, per metric × environment ──
# Tested on per-animal means (one value per mouse per environment), so a mouse
# contributing more than one session row doesn't pseudoreplicate the test — see
# Section 5 of CLAUDE.md. Both genotype groups must pass Shapiro (p > 0.05, n >= 3)
# for the metric × environment combination to be treated as normal. Levene's center
# is 'mean' (classic Levene) when normal, 'median' (Brown-Forsythe, robust to
# non-normality) otherwise. Both decisions drive the primary test choice in the
# next cell.
from scipy.stats import shapiro, levene

animal_agg_store  = {}   # (metric, environment) -> per-animal mean/median dataframe
normality_results = {}   # (metric, environment) -> bool (True = both genotypes normal)
levene_results    = {}   # (metric, environment) -> dict(statistic, p_value, center, equal_var)
diag_rows = []

for metric in metrics:
    for env in sorted(df["environment"].dropna().astype(str).unique()):
        sub = df.loc[df["environment"].astype(str) == env]
        if sub.empty:
            continue

        animal_df = (
            sub.groupby(["mouse_name", "Genotype"], observed=True)[metric]
            .agg(metric_mean="mean", metric_median="median", n_sessions="count")
            .reset_index()
        )
        animal_agg_store[(metric, env)] = animal_df

        shapiro_p = {}
        is_normal = True
        for geno in GENO_ORDER:
            vals = animal_df.loc[animal_df["Genotype"].astype(str) == geno, "metric_mean"].dropna().values
            if len(vals) < 3:
                is_normal = False
                shapiro_p[geno] = np.nan
            else:
                _, sw_p = shapiro(vals)
                shapiro_p[geno] = sw_p
                if sw_p <= 0.05:
                    is_normal = False
        normality_results[(metric, env)] = is_normal

        wt_vals   = animal_df.loc[animal_df["Genotype"].astype(str) == "WT",   "metric_mean"].dropna().values
        nlgf_vals = animal_df.loc[animal_df["Genotype"].astype(str) == "NLGF", "metric_mean"].dropna().values

        center = "mean" if is_normal else "median"
        if len(wt_vals) >= 2 and len(nlgf_vals) >= 2:
            lev_stat, lev_p = levene(wt_vals, nlgf_vals, center=center)
            equal_var = lev_p > 0.05
        else:
            lev_stat, lev_p, equal_var = np.nan, np.nan, np.nan
        levene_results[(metric, env)] = {
            "statistic": lev_stat, "p_value": lev_p,
            "center": center, "equal_var": equal_var,
        }

        diag_rows.append({
            "metric": metric, "environment": env,
            "n_WT_mice": len(wt_vals), "n_NLGF_mice": len(nlgf_vals),
            "shapiro_p_WT": shapiro_p.get("WT", np.nan),
            "shapiro_p_NLGF": shapiro_p.get("NLGF", np.nan),
            "normal": is_normal,
            "levene_center": center, "levene_stat": lev_stat, "levene_p": lev_p,
            "equal_var": equal_var,
        })

diagnostics_df = pd.DataFrame(diag_rows)
print(diagnostics_df.to_string(index=False))

                                 metric  environment  n_WT_mice  n_NLGF_mice  shapiro_p_WT  shapiro_p_NLGF  normal levene_center  levene_stat  levene_p  equal_var
            comodulogram_PAC_slow_gamma linear_track          8            9      0.564802        0.481402    True          mean     6.407032  0.023040      False
            comodulogram_PAC_slow_gamma   open_field          8            9      0.026968        0.786928   False        median     0.408014  0.532612       True
            comodulogram_PAC_fast_gamma linear_track          8            9      0.198971        0.014200   False        median     0.329108  0.574686       True
            comodulogram_PAC_fast_gamma   open_field          8            9      0.225016        0.032622   False        median     0.333269  0.572304       True
         n_oscillatory_epochs_slow_gamm linear_track          8            9      0.295565        0.070523    True          mean     1.861253  0.192601       True
         n_oscillatory

## Step 3 — Primary statistical test (Student's/Welch's t-test if normal, Mann-Whitney U otherwise), with BH-FDR correction

In [72]:
"""
Primary statistical test per metric × environment, chosen from the previous
cell's Shapiro (normality) and Levene's (variance-homogeneity) results:
  - normal combination  -> t-test on per-animal means (Student's if Levene's
    equal_var is True, Welch's otherwise)
  - non-normal combination -> Mann-Whitney U on per-animal medians
Benjamini-Hochberg FDR correction is then applied across all metric x
environment tests run in this cell (one family — these are all "is there a
genotype difference in this PAC/theta metric" questions, unlike the separate,
narrower FDR correction applied later to the GLMM-TMB vs secondary-test
concordance subset).
Saves:
  lfp_stats_outputs/
    primary_test_results.csv
    primary_test_results.html
"""

import os
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from IPython.display import display, HTML


def _pstars(p):
    if pd.isna(p): return ""
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return "ns"


test_results = []

for metric in metrics:
    for env in sorted(df["environment"].dropna().astype(str).unique()):
        key = (metric, env)
        if key not in animal_agg_store:
            continue

        animal_df = animal_agg_store[key]
        is_normal = normality_results[key]

        if is_normal:
            value_col = "metric_mean"
            equal_var = levene_results[key]["equal_var"]
            wt_vals   = animal_df.loc[animal_df["Genotype"].astype(str) == "WT",   value_col].dropna().values
            nlgf_vals = animal_df.loc[animal_df["Genotype"].astype(str) == "NLGF", value_col].dropna().values
            if len(wt_vals) < 2 or len(nlgf_vals) < 2:
                continue
            stat, p_val = ttest_ind(wt_vals, nlgf_vals, equal_var=bool(equal_var))
            test_name = "Student's t-test" if equal_var else "Welch's t-test"
        else:
            value_col = "metric_median"
            wt_vals   = animal_df.loc[animal_df["Genotype"].astype(str) == "WT",   value_col].dropna().values
            nlgf_vals = animal_df.loc[animal_df["Genotype"].astype(str) == "NLGF", value_col].dropna().values
            if len(wt_vals) < 1 or len(nlgf_vals) < 1:
                continue
            stat, p_val = mannwhitneyu(wt_vals, nlgf_vals, alternative="two-sided")
            test_name = "Mann-Whitney U"

        test_results.append({
            "metric": metric, "environment": env,
            "test": test_name, "value_col": value_col,
            "statistic": stat, "pvalue": p_val,
            "n_WT": len(wt_vals), "n_NLGF": len(nlgf_vals),
            "WT_value":   np.mean(wt_vals)   if is_normal else np.median(wt_vals),
            "NLGF_value": np.mean(nlgf_vals) if is_normal else np.median(nlgf_vals),
            "stars": _pstars(p_val),
        })

test_results_df = pd.DataFrame(test_results)

# ── Benjamini-Hochberg FDR correction across all metric x environment tests ────
ALPHA = 0.05
valid_p = test_results_df["pvalue"].notna()
test_results_df["pvalue_fdr"] = np.nan
test_results_df.loc[valid_p, "pvalue_fdr"] = multipletests(
    test_results_df.loc[valid_p, "pvalue"], method="fdr_bh"
)[1]
test_results_df["significant_fdr"] = test_results_df["pvalue_fdr"] < ALPHA
test_results_df["stars_fdr"] = test_results_df["pvalue_fdr"].apply(_pstars)

test_results_df.to_csv(os.path.join(OUTPUT_DIR, "primary_test_results.csv"), index=False)


def _colour_pval(p):
    if pd.isna(p): return ""
    if p < 0.001: return "background-color: #c62828; color: white; font-weight: bold;"
    if p < 0.01:  return "background-color: #ef5350; color: white; font-weight: bold;"
    if p < 0.05:  return "background-color: #ffcc80; color: black; font-weight: bold;"
    return ""


styled = (test_results_df.style
          .applymap(_colour_pval, subset=["pvalue", "pvalue_fdr"])
          .format({"statistic": "{:.4f}", "pvalue": "{:.4f}", "pvalue_fdr": "{:.4f}",
                   "WT_value": "{:.4f}", "NLGF_value": "{:.4f}"}))

display(HTML("<h3>Primary statistical test per metric × environment "
             "(t-test if normal, Mann-Whitney U otherwise), "
             "raw and Benjamini-Hochberg FDR-corrected p-values</h3>"))
display(styled)

html_path = os.path.join(OUTPUT_DIR, "primary_test_results.html")
with open(html_path, "w") as f:
    f.write(styled.to_html())

n_sig_raw = int(test_results_df["pvalue"].lt(ALPHA).sum())
n_sig_fdr = int(test_results_df["significant_fdr"].sum())
print(f"Significant at raw p<0.05: {n_sig_raw}/{len(test_results_df)}; "
      f"significant after BH-FDR: {n_sig_fdr}/{len(test_results_df)}")
print(f"✅ Primary test results saved to: {os.path.join(OUTPUT_DIR, 'primary_test_results.csv')}")
print(f"✅ HTML visualisation saved to: {html_path}")

/var/folders/7r/v13tprj17p5fvy167bbhtmch0000gn/T/ipykernel_4889/3524668634.py:97: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(_colour_pval, subset=["pvalue", "pvalue_fdr"])


,metric,environment,test,value_col,statistic,pvalue,n_WT,n_NLGF,WT_value,NLGF_value,stars,pvalue_fdr,significant_fdr,stars_fdr
0,comodulogram_PAC_slow_gamma,linear_track,Welch's t-test,metric_mean,3.2697,0.0116,8,9,0.0007,0.0002,*,0.0349,True,*
1,comodulogram_PAC_slow_gamma,open_field,Mann-Whitney U,metric_median,67.0000,0.0016,8,9,0.0007,0.0004,**,0.0063,True,**
2,comodulogram_PAC_fast_gamma,linear_track,Mann-Whitney U,metric_median,49.0000,0.2359,8,9,0.0015,0.0004,ns,0.3538,False,ns
3,comodulogram_PAC_fast_gamma,open_field,Mann-Whitney U,metric_median,49.0000,0.2359,8,9,0.0018,0.0006,ns,0.3538,False,ns
4,n_oscillatory_epochs_slow_gamm,linear_track,Student's t-test,metric_mean,-0.0111,0.9913,8,9,1347.2500,1349.0000,ns,0.9913,False,ns
5,n_oscillatory_epochs_slow_gamm,open_field,Welch's t-test,metric_mean,-1.0402,0.3194,8,9,2059.3750,2295.5556,ns,0.4259,False,ns
6,n_oscillatory_epochs_fast_gamma,linear_track,Student's t-test,metric_mean,0.5663,0.5796,8,9,2755.5000,2619.0000,ns,0.6955,False,ns
7,n_oscillatory_epochs_fast_gamma,open_field,Student's t-test,metric_mean,-0.0221,0.9827,8,9,4053.5000,4063.6667,ns,0.9913,False,ns
8,theta_gamma_modulation_index_fast_gamma,linear_track,Student's t-test,metric_mean,2.3026,0.0360,8,9,0.0640,0.0333,*,0.0865,False,ns
9,theta_gamma_modulation_index_fast_gamma,open_field,Student's t-test,metric_mean,1.5783,0.1353,8,9,0.0686,0.0409,ns,0.2707,False,ns


Significant at raw p<0.05: 5/12; significant after BH-FDR: 4/12
✅ Primary test results saved to: /Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis/primary_test_results.csv
✅ HTML visualisation saved to: /Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis/primary_test_results.html


## Generalized cell for all paired metrics with glmmTMB family selection:


- ✅ Loops over the paired metrics analysed here (PAC, n_oscillatory_epochs, theta_gamma_modulation_index)
- ✅ Per environment (open_field, linear_track)
- ✅ Family selection (Gaussian vs Gamma vs Inverse Gaussian via AIC; plus Beta, for `theta_gamma_modulation_index` only — it is bounded (0,1) by construction with no zeros in this dataset, unlike PAC which has exact zeros and `n_oscillatory_epochs` which is a raw count)
- ✅ Type III ANOVA + emmeans contrasts from best-fit family
- ✅ Organized output: Results saved to glmmTMB_paired_metrics/ subfolder
- ✅ Summary CSV: Shows which family won for each metric × environment combination

Results will include:

- {metric}_{env}_anova_{family}.csv
- {metric}_{env}_geno_by_band_{family}.csv
- {metric}_{env}_band_by_geno_{family}.csv
- {metric}_{env}_interaction_{family}.csv

In [74]:
# ── 2-way GLMM per environment for ALL PAIRED METRICS ────────────────────────
# Family selection: Gaussian vs Gamma vs Inverse Gaussian, plus Beta for
# theta_gamma_modulation_index only (bounded (0,1), no zeros in this dataset --
# unlike PAC, which has exact zeros, and n_oscillatory_epochs, which is a raw count)
# Metrics: PAC, n_oscillatory_epochs, theta_gamma_modulation_index (slow/fast variants)

import pandas as pd
import numpy as np
from pathlib import Path
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

ro.r("suppressPackageStartupMessages(library(glmmTMB))")
ro.r("suppressPackageStartupMessages(library(car))")
ro.r("suppressPackageStartupMessages(library(emmeans))")

def sig_label(p):
    if pd.isna(p): return 'NA'
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

# ── Define all paired metrics (slow/fast pairs) ────────────────────────────────
PAIRED_METRICS = {
    'PAC': {
        'slow': 'comodulogram_PAC_slow_gamma',
        'fast': 'comodulogram_PAC_fast_gamma',
        'label': 'Theta–gamma PAC',
    },
    'n_oscillatory_epochs': {
        'slow': 'n_oscillatory_epochs_slow_gamm',
        'fast': 'n_oscillatory_epochs_fast_gamma',
        'label': 'N oscillatory epochs',
    },
    'theta_gamma_modulation_index': {
        'slow': 'theta_gamma_modulation_index_slow_gamma',
        'fast': 'theta_gamma_modulation_index_fast_gamma',
        'label': 'Theta–gamma modulation index',
        # Bounded (0,1) by construction, no zeros observed -- Beta is a valid
        # candidate here only; not for PAC (has exact zeros) or
        # n_oscillatory_epochs (a raw count, not a proportion)
        'beta_eligible': True,
    },
}

# Filter to metrics that exist in df
PAIRED_METRICS = {k: v for k, v in PAIRED_METRICS.items() 
                  if v['slow'] in df.columns and v['fast'] in df.columns}

if not PAIRED_METRICS:
    print("No paired metrics found in data")
else:
    print(f"Found {len(PAIRED_METRICS)} paired metrics: {list(PAIRED_METRICS.keys())}")

# ── Master results storage ─────────────────────────────────────────────────────
all_paired_results = {}
family_summary_all = []

# ── Loop over each paired metric ────────────────────────────────────────────────
for metric_name, metric_info in PAIRED_METRICS.items():
    print(f"\n\n{'='*80}")
    print(f"METRIC: {metric_name} ({metric_info['label']})")
    print(f"{'='*80}")
    
    all_paired_results[metric_name] = {}
    
    # ── Loop over each environment ─────────────────────────────────────────
    for env in sorted(df['environment'].dropna().unique()):
        print(f"\n{'─'*80}\nENVIRONMENT: {env}\n{'─'*80}")
        
        # Subset to this environment
        sub_df = df[df['environment'] == env].copy()
        
        # Build long-format (slow + fast) for this metric
        slow = sub_df[['mouse_name', 'Genotype', metric_info['slow']]].rename(
            columns={metric_info['slow']: 'value'})
        slow['gamma_band'] = 'slow'
        
        fast = sub_df[['mouse_name', 'Genotype', metric_info['fast']]].rename(
            columns={metric_info['fast']: 'value'})
        fast['gamma_band'] = 'fast'
        
        long_data = pd.concat([slow, fast], ignore_index=True).dropna(subset=['value'])
        
        # Handle zeros/negatives: add small offset for Gamma/IG families
        min_positive = long_data.loc[long_data['value'] > 0, 'value'].min()
        offset = min_positive / 2 if pd.notna(min_positive) else 1e-5
        long_data['value_offset'] = long_data['value'].apply(lambda x: x if x > 0 else offset)
        
        long_data['mouse_name']  = long_data['mouse_name'].astype(str)
        long_data['Genotype']    = long_data['Genotype'].astype(str)
        long_data['gamma_band']  = long_data['gamma_band'].astype(str)
        
        print(f"n rows = {len(long_data)}, n mice = {long_data['mouse_name'].nunique()}")
        print(f"{metric_name} range: [{long_data['value'].min():.6f}, {long_data['value'].max():.6f}]")
        print(f"Offset for zeros: {offset:.2e}")
        print(long_data.groupby(['Genotype', 'gamma_band'])['value'].agg(['count', 'median']))
        
        if long_data['mouse_name'].nunique() < 4:
            print("  ⚠️  Skipped: insufficient data")
            continue
        
        # ── Pass to R and fit GLMM with family selection ─────────────────────
        with localconverter(ro.default_converter + pandas2ri.converter):
            r_df = ro.conversion.py2rpy(long_data)
        ro.globalenv['dat'] = r_df
        ro.globalenv['fit_beta_flag'] = metric_info.get('beta_eligible', False)
        
        r_code = """
        suppressWarnings({
            dat$mouse_name  <- factor(dat$mouse_name)
            dat$Genotype    <- factor(dat$Genotype,    levels = c("WT", "NLGF"))
            dat$gamma_band  <- factor(dat$gamma_band,  levels = c("slow", "fast"))
            
            contrasts(dat$Genotype)   <- contr.sum(nlevels(dat$Genotype))
            contrasts(dat$gamma_band) <- contr.sum(nlevels(dat$gamma_band))
            
            # ── Fit three families ──────────────────────────────────────────────
            fit_gaussian <- glmmTMB(
                value ~ Genotype * gamma_band + (1 | mouse_name),
                data = dat,
                family = gaussian()
            )
            
            fit_gamma <- glmmTMB(
                value_offset ~ Genotype * gamma_band + (1 | mouse_name),
                data = dat,
                family = Gamma(link = "log")
            )
            
            fit_igaussian <- glmmTMB(
                value_offset ~ Genotype * gamma_band + (1 | mouse_name),
                data = dat,
                family = inverse.gaussian(link = "log")
            )
            
            # Beta: only for metrics flagged beta-eligible (bounded (0,1), no zeros)
            aic_beta <- NA_real_
            if (isTRUE(fit_beta_flag)) {
                fit_beta <- glmmTMB(
                    value ~ Genotype * gamma_band + (1 | mouse_name),
                    data = dat,
                    family = beta_family(link = "logit")
                )
                aic_beta <- AIC(fit_beta)
            }
            
            # ── Compare AICs ───────────────────────────────────────────────────
            aic_gaussian   <- AIC(fit_gaussian)
            aic_gamma      <- AIC(fit_gamma)
            aic_igaussian  <- AIC(fit_igaussian)
            
            aic_list <- c(gaussian = aic_gaussian, gamma = aic_gamma, igaussian = aic_igaussian)
            if (isTRUE(fit_beta_flag)) aic_list <- c(aic_list, beta = aic_beta)
            best_family_name <- names(which.min(aic_list))
            best_fit <- switch(best_family_name,
                               gaussian  = fit_gaussian,
                               gamma     = fit_gamma,
                               igaussian = fit_igaussian,
                               beta      = fit_beta)
            
            delta_aic_min <- min(aic_list)
            delta_aic_from_best <- aic_list - delta_aic_min
            
            # ── Extract results from best fit via car::Anova (Type III) ─────────
            aov_table <- as.data.frame(Anova(best_fit, type = 3))
            aov_table <- data.frame(term = rownames(aov_table), aov_table,
                                    row.names = NULL, check.names = FALSE)
            
            # 1. Genotype contrast within each gamma band
            emm_geno_by_band <- emmeans(best_fit, ~ Genotype | gamma_band)
            geno_by_band_df  <- as.data.frame(contrast(emm_geno_by_band, method = "pairwise"))
            
            # 2. Gamma band contrast within each genotype
            emm_band_by_geno <- emmeans(best_fit, ~ gamma_band | Genotype)
            band_by_geno_df  <- as.data.frame(contrast(emm_band_by_geno, method = "pairwise"))
            
            # 3. Interaction contrast
            emm_full <- emmeans(best_fit, ~ Genotype * gamma_band)
            interaction_df <- as.data.frame(contrast(emm_full, interaction = "pairwise"))
            
            # Extract random effect variance and model fit info
            re_var  <- as.numeric(VarCorr(best_fit)$cond$mouse_name[1,1])
            aic_best <- AIC(best_fit)
            
            list(
                best_family        = best_family_name,
                aic_gaussian       = aic_gaussian,
                aic_gamma          = aic_gamma,
                aic_igaussian      = aic_igaussian,
                aic_beta           = aic_beta,
                delta_aics         = delta_aic_from_best,
                aic_best           = aic_best,
                anova_table        = aov_table,
                geno_by_band       = geno_by_band_df,
                band_by_geno       = band_by_geno_df,
                interaction        = interaction_df,
                re_var             = re_var
            )
        })
        """
        
        try:
            out = ro.r(r_code)
            
            with localconverter(ro.default_converter + pandas2ri.converter):
                aov_df          = ro.conversion.rpy2py(out.rx2('anova_table'))
                geno_by_band    = ro.conversion.rpy2py(out.rx2('geno_by_band'))
                band_by_geno    = ro.conversion.rpy2py(out.rx2('band_by_geno'))
                interaction_df  = ro.conversion.rpy2py(out.rx2('interaction'))
            
            best_family = str(out.rx2('best_family')[0])
            aic_gaussian = float(out.rx2('aic_gaussian')[0])
            aic_gamma = float(out.rx2('aic_gamma')[0])
            aic_igaussian = float(out.rx2('aic_igaussian')[0])
            aic_beta = float(out.rx2('aic_beta')[0])
            aic_best = float(out.rx2('aic_best')[0])
            re_var = float(out.rx2('re_var')[0])
            
            print(f"\n{'─'*70}")
            print(f"FAMILY SELECTION (Type III via car::Anova):")
            print(f"  Gaussian:        AIC = {aic_gaussian:.2f}")
            print(f"  Gamma:           AIC = {aic_gamma:.2f}")
            print(f"  Inverse Gaussian: AIC = {aic_igaussian:.2f}")
            if not np.isnan(aic_beta):
                print(f"  Beta:            AIC = {aic_beta:.2f}")
            print(f"  ✓ Best fit:      {best_family.upper()} (AIC = {aic_best:.2f})")
            print(f"{'─'*70}")
            
            print(f"\nModel fit (best family = {best_family}):")
            print(f"  AIC: {aic_best:.2f}  |  RE variance (mouse): {re_var:.6f}")
            
            print(f"\n─── Type III ANOVA ({best_family}) ───────────────────────────────────")
            print(aov_df.to_string(index=False))
            
            print(f"\n─── Genotype (NLGF vs WT) within each gamma band ────────")
            print(geno_by_band.to_string(index=False))
            
            print(f"\n─── Gamma band (fast vs slow) within each genotype ──────")
            print(band_by_geno.to_string(index=False))
            
            print(f"\n─── Interaction contrast ──────────────────────────────────")
            print(interaction_df.to_string(index=False))
            
            # Store for later reference
            all_paired_results[metric_name][env] = {
                'family': best_family,
                'aic_gaussian': aic_gaussian,
                'aic_gamma': aic_gamma,
                'aic_igaussian': aic_igaussian,
                'aic_beta': aic_beta,
                'anova': aov_df,
                'geno_by_band': geno_by_band,
                'band_by_geno': band_by_geno,
                'interaction': interaction_df,
            }
            
            family_summary_all.append({
                'metric': metric_name,
                'environment': env,
                'best_family': best_family,
                'aic_gaussian': aic_gaussian,
                'aic_gamma': aic_gamma,
                'aic_igaussian': aic_igaussian,
                'aic_beta': aic_beta,
                'n_mice': long_data['mouse_name'].nunique(),
            })
            
            # Save results
            env_clean = env.replace(" ", "_")
            metric_clean = metric_name.replace(" ", "_")
            out_dir = Path(OUTPUT_DIR) / 'glmmTMB_paired_metrics'
            out_dir.mkdir(parents=True, exist_ok=True)
            
            aov_df.to_csv(out_dir / f"{metric_clean}_{env_clean}_anova_{best_family}.csv", index=False)
            geno_by_band.to_csv(out_dir / f"{metric_clean}_{env_clean}_geno_by_band_{best_family}.csv", index=False)
            band_by_geno.to_csv(out_dir / f"{metric_clean}_{env_clean}_band_by_geno_{best_family}.csv", index=False)
            interaction_df.to_csv(out_dir / f"{metric_clean}_{env_clean}_interaction_{best_family}.csv", index=False)
            
            print(f"\n✅ Saved results for {metric_name} × {env} (family: {best_family})")
            
        except Exception as e:
            print(f"❌ FAILED: {e}")
            import traceback
            traceback.print_exc()

# ── Summary: families selected per metric × environment ────────────────────────
print(f"\n\n{'='*80}")
print("SUMMARY: FAMILIES SELECTED FOR ALL PAIRED METRICS")
print(f"{'='*80}")

if family_summary_all:
    summary_df = pd.DataFrame(family_summary_all)
    print(summary_df.to_string(index=False))
    
    # Save summary
    summary_path = Path(OUTPUT_DIR) / 'glmmTMB_paired_metrics' / "family_selection_summary_all_paired.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"\n✅ Family selection summary saved to: {summary_path}")

print(f"\n✅ All paired metrics analyzed and saved to: {Path(OUTPUT_DIR) / 'glmmTMB_paired_metrics'}")

Found 3 paired metrics: ['PAC', 'n_oscillatory_epochs', 'theta_gamma_modulation_index']


METRIC: PAC (Theta–gamma PAC)

────────────────────────────────────────────────────────────────────────────────
ENVIRONMENT: linear_track
────────────────────────────────────────────────────────────────────────────────
n rows = 34, n mice = 17
PAC range: [0.000043, 0.003230]
Offset for zeros: 2.16e-05
                     count    median
Genotype gamma_band                 
NLGF     fast            9  0.000445
         slow            9  0.000188
WT       fast            8  0.001543
         slow            8  0.000687

──────────────────────────────────────────────────────────────────────
FAMILY SELECTION (Type III via car::Anova):
  Gaussian:        AIC = -383.68
  Gamma:           AIC = -425.99
  Inverse Gaussian: AIC = -403.90
  ✓ Best fit:      GAMMA (AIC = -425.99)
──────────────────────────────────────────────────────────────────────

Model fit (best family = gamma):
  AIC: -425.99  |  RE v

In [75]:
# -- GLMM-TMB summary table: family selection + genotype effects, all paired metrics --
from pathlib import Path
from IPython.display import display, HTML
import numpy as np
import pandas as pd

GLMM_DIR = Path(OUTPUT_DIR) / "glmmTMB_paired_metrics"


def _anova_p(aov_df, term):
    if aov_df is None or "term" not in aov_df.columns or "Pr(>Chisq)" not in aov_df.columns:
        return np.nan
    row = aov_df.loc[aov_df["term"] == term, "Pr(>Chisq)"]
    return float(row.iloc[0]) if len(row) else np.nan


def _geno_by_band_p(gbb_df, band):
    if gbb_df is None or "gamma_band" not in gbb_df.columns or "p.value" not in gbb_df.columns:
        return np.nan
    row = gbb_df.loc[gbb_df["gamma_band"] == band, "p.value"]
    return float(row.iloc[0]) if len(row) else np.nan


rows = []
if "all_paired_results" in globals() and isinstance(all_paired_results, dict) and all_paired_results:
    for metric_name, env_dict in all_paired_results.items():
        label = (PAIRED_METRICS.get(metric_name, {}).get("label", metric_name)
                 if "PAIRED_METRICS" in globals() else metric_name)
        for env, res in env_dict.items():
            aov = res.get("anova")
            gbb = res.get("geno_by_band")
            rows.append({
                "metric": label,
                "environment": env,
                "best_family": res.get("family"),
                "aic_gaussian": res.get("aic_gaussian"),
                "aic_gamma": res.get("aic_gamma"),
                "aic_igaussian": res.get("aic_igaussian"),
                "aic_beta": res.get("aic_beta"),
                "Genotype_p": _anova_p(aov, "Genotype"),
                "gamma_band_p": _anova_p(aov, "gamma_band"),
                "Genotype_x_band_p": _anova_p(aov, "Genotype:gamma_band"),
                "Genotype_within_slow_p": _geno_by_band_p(gbb, "slow"),
                "Genotype_within_fast_p": _geno_by_band_p(gbb, "fast"),
            })
elif GLMM_DIR.exists():
    for path in sorted(GLMM_DIR.glob("*_anova_*.csv")):
        stem = path.name.removesuffix(".csv")
        metric_env, family = stem.rsplit("_anova_", 1)
        env = next((e for e in ("open_field", "linear_track") if metric_env.endswith(f"_{e}")), None)
        if env is None:
            continue
        metric_name = metric_env[: -(len(env) + 1)]
        aov = pd.read_csv(path)
        gbb_path = GLMM_DIR / f"{metric_name}_{env}_geno_by_band_{family}.csv"
        gbb = pd.read_csv(gbb_path) if gbb_path.exists() else None
        rows.append({
            "metric": metric_name,
            "environment": env,
            "best_family": family,
            "aic_gaussian": np.nan,
            "aic_gamma": np.nan,
            "aic_igaussian": np.nan,
            "aic_beta": np.nan,
            "Genotype_p": _anova_p(aov, "Genotype"),
            "gamma_band_p": _anova_p(aov, "gamma_band"),
            "Genotype_x_band_p": _anova_p(aov, "Genotype:gamma_band"),
            "Genotype_within_slow_p": _geno_by_band_p(gbb, "slow"),
            "Genotype_within_fast_p": _geno_by_band_p(gbb, "fast"),
        })
else:
    raise FileNotFoundError(
        "No GLMM-TMB results in memory or on disk -- run the GLMM-TMB cell above first."
    )

glmm_stats_table = pd.DataFrame(rows).sort_values(["environment", "metric"]).reset_index(drop=True)


def _colour_pval(p):
    if pd.isna(p): return ""
    if p < 0.001: return "background-color: #c62828; color: white; font-weight: bold;"
    if p < 0.01:  return "background-color: #ef5350; color: white; font-weight: bold;"
    if p < 0.05:  return "background-color: #ffcc80; color: black; font-weight: bold;"
    return ""


p_cols = ["Genotype_p", "gamma_band_p", "Genotype_x_band_p",
          "Genotype_within_slow_p", "Genotype_within_fast_p"]
aic_cols = ["aic_gaussian", "aic_gamma", "aic_igaussian", "aic_beta"]

display(HTML("<h3>GLMM-TMB family selection and genotype effects — all paired metrics</h3>"))
display(
    glmm_stats_table.style
    .applymap(_colour_pval, subset=p_cols)
    .format({**{c: "{:.2f}" for c in aic_cols}, **{c: "{:.4f}" for c in p_cols}}, na_rep="—")
    .set_properties(**{"font-size": "11px"})
)

summary_out_path = GLMM_DIR / "glmmTMB_family_and_genotype_effects_summary.csv"
glmm_stats_table.to_csv(summary_out_path, index=False)
print(f"Saved: {summary_out_path}")


/var/folders/7r/v13tprj17p5fvy167bbhtmch0000gn/T/ipykernel_4889/1913796511.py:94: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(_colour_pval, subset=p_cols)


,metric,environment,best_family,aic_gaussian,aic_gamma,aic_igaussian,aic_beta,Genotype_p,gamma_band_p,Genotype_x_band_p,Genotype_within_slow_p,Genotype_within_fast_p
0,N oscillatory epochs,linear_track,igaussian,496.07,486.55,483.64,—,0.6405,0.0000,0.1590,0.9578,0.3770
1,Theta–gamma PAC,linear_track,gamma,-383.68,-425.99,-403.90,—,0.0028,0.0002,0.2857,0.0018,0.1262
2,Theta–gamma modulation index,linear_track,beta,-158.21,-164.25,-158.57,-164.52,0.0000,0.8382,0.9325,0.0032,0.0019
3,N oscillatory epochs,open_field,gamma,549.67,538.47,—,—,0.6234,0.0000,0.2990,0.3589,0.9618
4,Theta–gamma PAC,open_field,gamma,-357.81,-398.67,-378.76,—,0.0213,0.0001,0.5750,0.0429,0.2179
5,Theta–gamma modulation index,open_field,beta,-142.46,-146.71,-142.57,-148.35,0.0015,0.7771,0.7648,0.0385,0.0147


Saved: /Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis/glmmTMB_paired_metrics/glmmTMB_family_and_genotype_effects_summary.csv


## Plot the graphs — per environment, slow and fast gamma together

One figure per environment; each subplot is a paired metric (PAC, n_oscillatory_epochs, theta_gamma_modulation_index) with slow and fast gamma shown together, grouped by band and colored by genotype. Annotated with the GLMM-TMB Type III ANOVA Genotype and gamma_band main-effect p-values, plus the within-band WT vs NLGF contrasts (slow, fast) as brackets, all from the paired-metrics cell above (`all_paired_results`) -- not the Step 3 per-column secondary test used by the previous version of this plot.

In [ ]:
# Per-environment plots for the paired metrics (PAC, n_oscillatory_epochs,
# theta_gamma_modulation_index): one figure per environment, one subplot per
# paired metric, slow and fast gamma shown together within each subplot
# (grouped by band, boxes colored by genotype) instead of the previous
# one-figure-per-metric/side-by-side-environments layout.
# Annotated with the GLMM-TMB Type III ANOVA Genotype and gamma_band
# main-effect p-values, plus the within-band WT vs NLGF contrasts (slow, fast)
# from the same paired-metrics cell's emmeans output, not the Step 3
# per-column secondary test.
# Run after Step 1-3 (animal_agg_store, test_results_df) and the GLMM-TMB
# paired-metrics cell (all_paired_results, PAIRED_METRICS).

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PAIRED_PLOT_OUTDIR = os.path.join(OUTPUT_DIR, "glmmTMB_paired_metrics", "per_environment_plots")
os.makedirs(PAIRED_PLOT_OUTDIR, exist_ok=True)

preferred_env_order = ["open_field", "linear_track"]
envs_present = list(df["environment"].dropna().astype(str).unique())
envs = [e for e in preferred_env_order if e in envs_present]
envs += [e for e in sorted(envs_present) if e not in envs]

BAND_X = {"slow": 0.0, "fast": 1.6}
GENO_OFFSET = {"WT": -0.32, "NLGF": 0.32}


def _clean_name(x):
    return str(x).replace("/", "_").replace(" ", "_")


def _anova_p(env_results, term):
    aov = env_results.get("anova") if env_results else None
    if aov is None or "term" not in aov.columns or "Pr(>Chisq)" not in aov.columns:
        return np.nan
    row = aov.loc[aov["term"] == term, "Pr(>Chisq)"]
    return float(row.iloc[0]) if len(row) else np.nan


def _stars(p):
    if pd.isna(p): return "ns"
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"


def _p_line(p):
    if pd.isna(p):
        return "n/a"
    return "p<0.001" if p < 0.001 else f"p={p:.3f}"


def _ptext(label, p):
    if pd.isna(p):
        return f"{label}: n/a"
    return f"{label} {_stars(p)} ({_p_line(p)})"


def _value_col_for(col, env):
    row = test_results_df[(test_results_df["metric"] == col) & (test_results_df["environment"] == env)]
    return row.iloc[0]["value_col"] if not row.empty else "metric_mean"


def _geno_by_band_p(env_results, band):
    gbb = env_results.get("geno_by_band") if env_results else None
    if gbb is None or "gamma_band" not in gbb.columns or "p.value" not in gbb.columns:
        return np.nan
    row = gbb.loc[gbb["gamma_band"] == band, "p.value"]
    return float(row.iloc[0]) if len(row) else np.nan


def _draw_bracket(ax, x1, x2, y, h, label):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y],
            color="black", linewidth=0.7, clip_on=False, zorder=12)
    ax.text((x1 + x2) / 2, y + h, label, ha="center", va="bottom", fontsize=5.5, zorder=12)


def plot_paired_metrics_for_environment(env, save=True):
    metrics_for_env = [
        (name, info) for name, info in PAIRED_METRICS.items()
        if env in all_paired_results.get(name, {})
    ]
    if not metrics_for_env:
        return

    fig, axes = plt.subplots(1, len(metrics_for_env), figsize=(2.6 * len(metrics_for_env), 2.6))
    if len(metrics_for_env) == 1:
        axes = [axes]

    rng = np.random.default_rng(10)

    for ax, (metric_name, info) in zip(axes, metrics_for_env):
        env_results = all_paired_results[metric_name][env]

        band_vals = {}
        for band, col in (("slow", info["slow"]), ("fast", info["fast"])):
            key = (col, env)
            if key not in animal_agg_store:
                continue
            value_col = _value_col_for(col, env)
            animal_df = animal_agg_store[key]
            band_vals[band] = {
                g: animal_df.loc[animal_df["Genotype"].astype(str) == g, value_col].dropna().values
                for g in GENO_ORDER if g in animal_df["Genotype"].astype(str).values
            }

        all_y = np.concatenate([
            v for band in band_vals.values() for v in band.values() if len(v)
        ]) if band_vals else np.array([])
        if all_y.size == 0:
            ax.set_visible(False)
            continue
        ymin, ymax = np.nanmin(all_y), np.nanmax(all_y)
        yrng = ymax - ymin if ymax > ymin else max(abs(ymax), 1.0) * 0.1
        ax.set_ylim(ymin - 0.10 * yrng, ymax + 0.70 * yrng)

        xticks, xticklabels = [], []
        for band, bx in BAND_X.items():
            genos = band_vals.get(band, {})
            for g in GENO_ORDER:
                if g not in genos or len(genos[g]) == 0:
                    continue
                x = bx + GENO_OFFSET[g]
                y = genos[g]
                ax.boxplot(
                    y, positions=[x], widths=0.28,
                    patch_artist=True, manage_ticks=False, showfliers=False,
                    boxprops=dict(facecolor=GENO_COLORS.get(g, "#aaaaaa"), alpha=0.55, linewidth=0.5),
                    medianprops=dict(color="black", linewidth=0.8),
                    whiskerprops=dict(linewidth=0.5),
                    capprops=dict(linewidth=0.5),
                    zorder=7,
                )
                jitter = rng.normal(0, 0.03, size=len(y))
                ax.scatter(np.full(len(y), x) + jitter, y,
                           s=12, color=GENO_COLORS.get(g, "#aaaaaa"),
                           edgecolor="black", linewidth=0.3, alpha=0.9, zorder=10)
                xticks.append(x)
                xticklabels.append(f"{g}\n(n={len(y)})")

        # Within-band WT vs NLGF brackets (GLMM-TMB emmeans pairwise contrast)
        for band, bx in BAND_X.items():
            genos = band_vals.get(band, {})
            if "WT" not in genos or "NLGF" not in genos or len(genos["WT"]) == 0 or len(genos["NLGF"]) == 0:
                continue
            band_max = max(np.nanmax(genos["WT"]), np.nanmax(genos["NLGF"]))
            p_band = _geno_by_band_p(env_results, band)
            _draw_bracket(
                ax,
                bx + GENO_OFFSET["WT"], bx + GENO_OFFSET["NLGF"],
                band_max + 0.06 * yrng, 0.035 * yrng,
                f"{_stars(p_band)}\n{_p_line(p_band)}",
            )

        ax.set_xticks(xticks)
        ax.set_xticklabels(xticklabels, fontsize=5.5)
        ax.set_xlim(BAND_X["slow"] - 0.7, BAND_X["fast"] + 0.7)

        # Band-group headers (slow / fast), below the genotype tick labels
        for band, bx in BAND_X.items():
            if band in band_vals and any(len(v) for v in band_vals[band].values()):
                ax.annotate(
                    band, xy=(bx, 0), xycoords=("data", "axes fraction"),
                    xytext=(0, -22), textcoords="offset points",
                    ha="center", va="top", fontsize=6, fontweight="bold", clip_on=False,
                )

        # GLMM-TMB Type III main-effect annotations (Genotype, gamma_band),
        # placed above the within-band brackets
        geno_p = _anova_p(env_results, "Genotype")
        band_p = _anova_p(env_results, "gamma_band")
        ax.text(
            0.5, ymax + 0.34 * yrng,
            f"{_ptext('Genotype', geno_p)}\n{_ptext('Band', band_p)}",
            transform=ax.get_yaxis_transform(), ha="center", va="bottom", fontsize=5.5,
        )

        set_panel_title(ax, info["label"])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(env.replace("_", " "), fontsize=8, fontweight="bold")
    fig.tight_layout(rect=[0, 0.06, 1, 0.92])

    if save:
        fig.savefig(os.path.join(
            PAIRED_PLOT_OUTDIR, f"{_clean_name(env)}_paired_metrics_slow_fast_glmm.pdf"
        ))

    plt.show()
    plt.close(fig)


for env in envs:
    plot_paired_metrics_for_environment(env)

print("Done.")
print(f"Saved per-environment paired-metric plots to: {PAIRED_PLOT_OUTDIR}")


## Compare GLMM-TMB vs Mann-Whitney U: concordance of genotype effects


This comparison uses the environment-specific GLMM-TMB genotype contrasts and compares them with Mann-Whitney U genotype tests for the same environment and original metric column.

For paired slow/fast metrics, GLMM-TMB rows are mapped back to the matching original slow/fast columns before merging with Mann-Whitney results.


In [76]:
# -- Compare GLMM-TMB vs Step 3 secondary test: genotype contrast concordance -----
# Step 3 (cell 6/7) picks a per-metric x environment secondary test (t-test/Welch's
# when normal, Mann-Whitney U otherwise), so this is no longer always MWU -- see
# the 'secondary_test' column below for which test backs each row.
from pathlib import Path
from IPython.display import display, HTML
import os
import numpy as np
import pandas as pd

ALPHA = 0.05
GLMM_DIR = Path(OUTPUT_DIR) / "glmmTMB_paired_metrics"
SECONDARY_PATH = Path(OUTPUT_DIR) / "primary_test_results.csv"

PAIRED_ORIGINAL_METRIC = {
    ("PAC", "slow"): "comodulogram_PAC_slow_gamma",
    ("PAC", "fast"): "comodulogram_PAC_fast_gamma",
    ("n_oscillatory_epochs", "slow"): "n_oscillatory_epochs_slow_gamm",
    ("n_oscillatory_epochs", "fast"): "n_oscillatory_epochs_fast_gamma",
    ("theta_gamma_modulation_index", "slow"): "theta_gamma_modulation_index_slow_gamma",
    ("theta_gamma_modulation_index", "fast"): "theta_gamma_modulation_index_fast_gamma",
}

PRETTY_METRIC = {
    "comodulogram_PAC_slow_gamma": "PAC slow",
    "comodulogram_PAC_fast_gamma": "PAC fast",
    "n_oscillatory_epochs_slow_gamm": "N oscillatory epochs slow",
    "n_oscillatory_epochs_fast_gamma": "N oscillatory epochs fast",
    "theta_gamma_modulation_index_slow_gamma": "Theta–Slow gamma modulation index",
    "theta_gamma_modulation_index_fast_gamma": "Theta–Fast gamma modulation index",
}

def sig_label(p):
    if pd.isna(p): return "NA"
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def clean_key(value):
    return str(value).strip().lower().replace(" ", "_").replace("/", "_")

def p_value_column(table):
    for candidate in ["p.value", "pvalue", "Pr(>|z|)", "Pr(>|t|)", "p"]:
        if candidate in table.columns:
            return candidate
    candidates = [col for col in table.columns if "p" in str(col).lower()]
    return candidates[0] if candidates else None

def stat_column(table):
    for candidate in ["z.ratio", "t.ratio", "statistic"]:
        if candidate in table.columns:
            return candidate
    return None

def parse_geno_filename(path, env_lookup):
    """Parse {metric}_{environment}_geno_by_band_{family}.csv."""
    stem = path.name.removesuffix(".csv")
    if "_geno_by_band_" not in stem:
        return None

    for env_clean, env_label in sorted(env_lookup.items(), key=lambda item: len(item[0]), reverse=True):
        marker = f"_{env_clean}_geno_by_band_"
        if marker in stem:
            metric_name, family = stem.split(marker, 1)
            return metric_name, env_label, family
    return None

def extract_glmm_rows_from_table(table, metric_name, environment, family):
    rows = []
    p_col = p_value_column(table)
    s_col = stat_column(table)
    if p_col is None:
        return rows

    for _, row in table.iterrows():
        if "gamma_band" in table.columns and pd.notna(row.get("gamma_band")):
            gamma_band = str(row.get("gamma_band")).strip().lower()
        else:
            contrast_text = str(row.get("contrast", "")).lower()
            if "slow" in contrast_text:
                gamma_band = "slow"
            elif "fast" in contrast_text:
                gamma_band = "fast"
            else:
                continue

        original_metric = PAIRED_ORIGINAL_METRIC.get((metric_name, gamma_band))
        if original_metric is None:
            continue

        rows.append({
            "metric_group": metric_name,
            "gamma_band": gamma_band,
            "original_metric": original_metric,
            "metric": PRETTY_METRIC.get(original_metric, original_metric),
            "environment": environment,
            "environment_key": clean_key(environment),
            "glmm_family": family,
            "glmm_contrast": row.get("contrast", np.nan),
            "glmm_estimate": row.get("estimate", np.nan),
            "glmm_SE": row.get("SE", np.nan),
            "glmm_df": row.get("df", np.nan),
            "glmm_stat": row.get(s_col, np.nan) if s_col is not None else np.nan,
            "glmm_p": pd.to_numeric(row.get(p_col), errors="coerce"),
        })
    return rows

def extract_glmm_rows():
    rows = []

    if "all_paired_results" in globals() and isinstance(all_paired_results, dict) and all_paired_results:
        for metric_name, env_dict in all_paired_results.items():
            for environment, env_data in env_dict.items():
                table = env_data.get("geno_by_band")
                if table is None or len(table) == 0:
                    continue
                family = env_data.get("family", np.nan)
                rows.extend(extract_glmm_rows_from_table(table, metric_name, environment, family))

    if rows:
        return pd.DataFrame(rows)

    if not GLMM_DIR.exists():
        raise FileNotFoundError(f"GLMM-TMB results folder not found: {GLMM_DIR}")

    env_lookup = {clean_key(env): env for env in sorted(df["environment"].dropna().unique())}
    for path in sorted(GLMM_DIR.glob("*_geno_by_band_*.csv")):
        parsed = parse_geno_filename(path, env_lookup)
        if parsed is None:
            print(f"Skipping unrecognized GLMM filename: {path.name}")
            continue
        metric_name, environment, family = parsed
        table = pd.read_csv(path)
        rows.extend(extract_glmm_rows_from_table(table, metric_name, environment, family))

    return pd.DataFrame(rows)

# Load Step 3 secondary-test results from memory when available, otherwise from disk.
if "test_results_df" in globals() and isinstance(test_results_df, pd.DataFrame):
    secondary_df = test_results_df.copy()
elif SECONDARY_PATH.exists():
    secondary_df = pd.read_csv(SECONDARY_PATH)
else:
    raise FileNotFoundError(f"Step 3 secondary-test results not found: {SECONDARY_PATH}")

glmm_df = extract_glmm_rows()
if glmm_df.empty:
    raise ValueError("No GLMM-TMB genotype-by-gamma-band contrast rows were extracted.")

required_secondary_cols = [
    "metric", "environment", "test", "n_WT", "n_NLGF", "WT_value", "NLGF_value", "statistic", "pvalue"
]
missing = [col for col in required_secondary_cols if col not in secondary_df.columns]
if missing:
    raise KeyError(f"Step 3 secondary-test table is missing required columns: {missing}")

secondary_for_comparison = (
    secondary_df.loc[secondary_df["metric"].isin(PAIRED_ORIGINAL_METRIC.values()), required_secondary_cols]
    .rename(columns={"metric": "original_metric", "test": "secondary_test",
                      "statistic": "secondary_statistic", "pvalue": "secondary_p"})
    .copy()
)
secondary_for_comparison["environment_key"] = secondary_for_comparison["environment"].map(clean_key)

comp = pd.merge(
    glmm_df,
    secondary_for_comparison,
    on=["original_metric", "environment_key"],
    how="inner",
    suffixes=("", "_secondary"),
)

if comp.empty:
    raise ValueError(
        "No GLMM-TMB rows matched Mann-Whitney rows. Check environment labels and metric names."
    )

# Keep GLMM environment label and drop duplicate secondary-test environment label.
if "environment_secondary" in comp.columns:
    comp = comp.drop(columns=["environment_secondary"])

comp["glmm_sig"] = comp["glmm_p"] < ALPHA
comp["secondary_sig"] = comp["secondary_p"] < ALPHA
comp["glmm_sig_label"] = comp["glmm_p"].map(sig_label)
comp["secondary_sig_label"] = comp["secondary_p"].map(sig_label)

def classify(row):
    if row["glmm_sig"] and row["secondary_sig"]:
        return "both_significant"
    if row["glmm_sig"] and not row["secondary_sig"]:
        return "glmmTMB_only"
    if (not row["glmm_sig"]) and row["secondary_sig"]:
        return "secondary_only"
    return "both_not_significant"

comp["concordance"] = comp.apply(classify, axis=1)
comp = comp.sort_values(["environment", "metric_group", "gamma_band"]).reset_index(drop=True)

full_path = Path(OUTPUT_DIR) / "glmmTMB_vs_secondary_test_concordance_by_gamma_band.csv"
sig_path = Path(OUTPUT_DIR) / "glmmTMB_vs_secondary_test_significant_results_by_gamma_band.csv"
comp.to_csv(full_path, index=False)
comp.loc[comp["glmm_sig"] | comp["secondary_sig"]].to_csv(sig_path, index=False)

print(f"Extracted GLMM-TMB rows: {len(glmm_df)}")
print(f"Matched comparisons: {len(comp)}")
print(f"Saved full concordance table: {full_path}")
print(f"Saved significant-results table: {sig_path}")

print("\nConcordance counts:")
print(comp["concordance"].value_counts(dropna=False).to_string())

print("\nConcordance by environment:")
print(pd.crosstab(comp["environment"], comp["concordance"]).to_string())

print("\nConcordance by gamma band:")
print(pd.crosstab(comp["gamma_band"], comp["concordance"]).to_string())

show_cols = [
    "metric", "environment", "gamma_band", "glmm_family",
    "glmm_estimate", "glmm_p", "glmm_sig_label",
    "secondary_test", "secondary_p", "secondary_sig_label", "secondary_statistic",
    "WT_value", "NLGF_value", "n_WT", "n_NLGF", "concordance",
]

display(HTML("<h3>GLMM-TMB vs Step 3 secondary test: genotype-effect concordance</h3>"))
display(
    comp[show_cols].style
    .format({
        "glmm_estimate": "{:.6f}",
        "glmm_p": "{:.4f}",
        "secondary_p": "{:.4f}",
        "secondary_statistic": "{:.2f}",
        "WT_value": "{:.6f}",
        "NLGF_value": "{:.6f}",
    })
    .set_properties(**{"font-size": "11px"})
)

glmmTMB_vs_secondary_test = comp


Extracted GLMM-TMB rows: 12
Matched comparisons: 12
Saved full concordance table: /Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis/glmmTMB_vs_secondary_test_concordance_by_gamma_band.csv
Saved significant-results table: /Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis/glmmTMB_vs_secondary_test_significant_results_by_gamma_band.csv

Concordance counts:
concordance
both_not_significant    6
both_significant        5
glmmTMB_only            1

Concordance by environment:
concordance   both_not_significant  both_significant  glmmTMB_only
environment                                                       
linear_track                     3                 3             0
open_field                       3                 2             1

Concordance by gamma band:
concordance  both_not_significant  both_significant  glmmTMB_on

,metric,environment,gamma_band,glmm_family,glmm_estimate,glmm_p,glmm_sig_label,secondary_test,secondary_p,secondary_sig_label,secondary_statistic,WT_value,NLGF_value,n_WT,n_NLGF,concordance
0,PAC fast,linear_track,fast,gamma,0.696273,0.1262,ns,Mann-Whitney U,0.2359,ns,49.00,0.001543,0.000445,8,9,both_not_significant
1,PAC slow,linear_track,slow,gamma,1.293371,0.0018,**,Welch's t-test,0.0116,*,3.27,0.000675,0.000189,8,9,both_significant
2,N oscillatory epochs fast,linear_track,fast,igaussian,0.073455,0.3770,ns,Student's t-test,0.5796,ns,0.57,2755.500000,2619.000000,8,9,both_not_significant
3,N oscillatory epochs slow,linear_track,slow,igaussian,0.004831,0.9578,ns,Student's t-test,0.9913,ns,-0.01,1347.250000,1349.000000,8,9,both_not_significant
4,Theta–Fast gamma modulation index,linear_track,fast,beta,0.691334,0.0019,**,Student's t-test,0.0360,*,2.30,0.064006,0.033314,8,9,both_significant
5,Theta–Slow gamma modulation index,linear_track,slow,beta,0.664538,0.0032,**,Mann-Whitney U,0.0010,***,68.00,0.060694,0.024642,8,9,both_significant
6,PAC fast,open_field,fast,gamma,0.504388,0.2179,ns,Mann-Whitney U,0.2359,ns,49.00,0.001761,0.000555,8,9,both_not_significant
7,PAC slow,open_field,slow,gamma,0.829007,0.0429,*,Mann-Whitney U,0.0016,**,67.00,0.000703,0.000363,8,9,both_significant
8,N oscillatory epochs fast,open_field,fast,gamma,0.004906,0.9618,ns,Student's t-test,0.9827,ns,-0.02,4053.500000,4063.666667,8,9,both_not_significant
9,N oscillatory epochs slow,open_field,slow,gamma,-0.094066,0.3589,ns,Welch's t-test,0.3194,ns,-1.04,2059.375000,2295.555556,8,9,both_not_significant


## FDR Correction Across MLM vs Secondary-Test Results

The concordance table above compares raw p-values across metric × environment combinations (12 tests here). Running that many tests without correction inflates the false-positive rate — a handful of p-values in the 0.04-0.06 range are expected by chance alone.

This cell applies **Benjamini-Hochberg FDR correction** separately to the MLM p-values and the Step 3 secondary-test p-values (t-test/Welch's or Mann-Whitney U, per metric × environment — each family corrected on its own, since they are different test families answering different statistical questions), then re-classifies concordance using the corrected significance calls.

**Interpretation:**
- `both_fdr` — significant in both tests after correction → the most trustworthy findings
- `GLMM-TMB_only_fdr` / `secondary_only_fdr` — significant in one test but not the other after correction → treat as borderline; check for outlier-driven effects (e.g. via the animal-level violin plots) or run a robustness check (e.g. median-based permutation test)
- `neither_fdr` — not significant in either after correction

Re-run this cell any time `comp` is rebuilt (e.g. after adding new metrics).

In [77]:
# ── FDR correction (Benjamini-Hochberg) across all GLMM-TMB vs secondary-test results ───────────
# Corrects for running one test per metric × environment combination.
# Adds *_p_fdr and *_sig_fdr columns to `comp`, plus a corrected concordance
# label. Depends on `comp` from the concordance cell above — re-run this any
# time `comp` is rebuilt (e.g. after adding new metrics).

from statsmodels.stats.multitest import multipletests

ALPHA = 0.05

def apply_fdr_correction(comp_df, alpha=ALPHA):
    """
    Applies Benjamini-Hochberg FDR correction separately to the GLMM-TMB and Step 3
    secondary-test (t-test/Welch's or Mann-Whitney U, per metric x environment)
    p-value columns of the concordance table, and re-classifies concordance
    using the corrected p-values.

    The two families (GLMM-TMB, secondary test) are corrected independently, since
    they are different test types answering different statistical questions —
    pooling them into one correction would not be appropriate.
    """
    comp_df = comp_df.copy()

    # GLMM-TMB correction
    glmm_mask = comp_df['glmm_p'].notna()
    comp_df.loc[glmm_mask, 'glmm_p_fdr'] = multipletests(
        comp_df.loc[glmm_mask, 'glmm_p'], method='fdr_bh'
    )[1]
    comp_df['glmm_sig_fdr'] = comp_df['glmm_p_fdr'] < alpha

    # Secondary-test correction
    secondary_mask = comp_df['secondary_p'].notna()
    comp_df.loc[secondary_mask, 'secondary_p_fdr'] = multipletests(
        comp_df.loc[secondary_mask, 'secondary_p'], method='fdr_bh'
    )[1]
    comp_df['secondary_sig_fdr'] = comp_df['secondary_p_fdr'] < alpha

    # Re-classify concordance on FDR-corrected p-values
    def classify_fdr(row):
        g = bool(row['glmm_sig_fdr']) if pd.notna(row.get('glmm_sig_fdr')) else False
        s = bool(row['secondary_sig_fdr']) if pd.notna(row.get('secondary_sig_fdr')) else False
        if g and s:
            return 'both_fdr'
        if g:
            return 'GLMM-TMB_only_fdr'
        if s:
            return 'secondary_only_fdr'
        return 'neither_fdr'

    comp_df['concordance_fdr'] = comp_df.apply(classify_fdr, axis=1)
    return comp_df


comp_fdr = apply_fdr_correction(comp)

display(HTML("<h3>GLMM-TMB vs Step 3 secondary test — after Benjamini-Hochberg FDR correction</h3>"))
show_cols = ['metric', 'environment',
             'glmm_p', 'glmm_p_fdr', 'glmm_sig_fdr',
             'secondary_p', 'secondary_p_fdr', 'secondary_sig_fdr',
             'concordance', 'concordance_fdr']

def _colour_sig(row):
    g_sig = bool(row.get('glmm_sig_fdr'))
    s_sig = bool(row.get('secondary_sig_fdr'))
    if g_sig and s_sig:
        bg = "#c8e6c9"   # green — robust
    elif g_sig or s_sig:
        bg = "#fff9c4"   # yellow — borderline / method-dependent
    else:
        bg = ""
    return [f"background-color:{bg}" if bg else "" for _ in row]

display(
    comp_fdr[show_cols]
        .sort_values(['environment', 'metric'])
        .style
        .apply(_colour_sig, axis=1)
        .format({
            'glmm_p':     '{:.4f}',
            'glmm_p_fdr': '{:.4f}',
            'secondary_p':     '{:.4f}',
            'secondary_p_fdr': '{:.4f}',
        })
        .set_properties(**{'font-size': '11px'})
)

print("\n── Concordance counts BEFORE FDR correction ────────────────────────────")
print(comp_fdr['concordance'].value_counts().to_string())
print("\n── Concordance counts AFTER FDR correction ─────────────────────────────")
print(comp_fdr['concordance_fdr'].value_counts().to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
fdr_path = os.path.join(OUTPUT_DIR, "glmm-tmb_vs_secondary_test_concordance_FDR.csv")
comp_fdr.to_csv(fdr_path, index=False)
print(f"\n✅ Saved: {fdr_path}")

,metric,environment,glmm_p,glmm_p_fdr,glmm_sig_fdr,secondary_p,secondary_p_fdr,secondary_sig_fdr,concordance,concordance_fdr
2,N oscillatory epochs fast,linear_track,0.3770,0.4524,False,0.5796,0.6955,False,both_not_significant,neither_fdr
3,N oscillatory epochs slow,linear_track,0.9578,0.9618,False,0.9913,0.9913,False,both_not_significant,neither_fdr
0,PAC fast,linear_track,0.1262,0.2164,False,0.2359,0.3538,False,both_not_significant,neither_fdr
1,PAC slow,linear_track,0.0018,0.0115,True,0.0116,0.0349,True,both_significant,both_fdr
4,Theta–Fast gamma modulation index,linear_track,0.0019,0.0115,True,0.0360,0.0865,False,both_significant,GLMM-TMB_only_fdr
5,Theta–Slow gamma modulation index,linear_track,0.0032,0.0128,True,0.0010,0.0059,True,both_significant,both_fdr
8,N oscillatory epochs fast,open_field,0.9618,0.9618,False,0.9827,0.9913,False,both_not_significant,neither_fdr
9,N oscillatory epochs slow,open_field,0.3589,0.4524,False,0.3194,0.4259,False,both_not_significant,neither_fdr
6,PAC fast,open_field,0.2179,0.3268,False,0.2359,0.3538,False,both_not_significant,neither_fdr
7,PAC slow,open_field,0.0429,0.0857,False,0.0016,0.0063,True,both_significant,secondary_only_fdr



── Concordance counts BEFORE FDR correction ────────────────────────────
concordance
both_not_significant    6
both_significant        5
glmmTMB_only            1

── Concordance counts AFTER FDR correction ─────────────────────────────
concordance_fdr
neither_fdr           6
both_fdr              2
GLMM-TMB_only_fdr     2
secondary_only_fdr    2

✅ Saved: /Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis/glmm-tmb_vs_secondary_test_concordance_FDR.csv


## Median-based permutation test Apply your median-based permutation test 
-specific discordant cells as a tiebreaker 
— it'll tell you whether the GLMM or the Step 3 secondary test is closer to the "outlier-robust truth."

In [78]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd

ALPHA = 0.05

base = Path(
    "/Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/"
    "DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis"
)

data_path = base / "concatenated_lfp_stats.csv"
concordance_path = base / "glmmTMB_vs_secondary_test_concordance_by_gamma_band.csv"
out_path = base / "median_permutation_tiebreaker_discordant_glmmTMB_vs_secondary_test.csv"

lfp = pd.read_csv(data_path)
comp = pd.read_csv(concordance_path)

discordant = comp[comp["concordance"].isin(["glmmTMB_only", "secondary_only"])].copy()


def exact_median_permutation(values, labels, wt_label="WT", ko_label="NLGF"):
    values = np.asarray(values, dtype=float)
    labels = np.asarray(labels).astype(str)

    keep = np.isfinite(values) & np.isin(labels, [wt_label, ko_label])
    values = values[keep]
    labels = labels[keep]

    n_total = len(values)
    n_wt = int(np.sum(labels == wt_label))
    n_ko = int(np.sum(labels == ko_label))

    if n_wt == 0 or n_ko == 0:
        return None

    obs = float(
        np.nanmedian(values[labels == wt_label])
        - np.nanmedian(values[labels == ko_label])
    )
    abs_obs = abs(obs)

    diffs = []
    for wt_idx in combinations(range(n_total), n_wt):
        wt_mask = np.zeros(n_total, dtype=bool)
        wt_mask[list(wt_idx)] = True

        diff = float(
            np.nanmedian(values[wt_mask])
            - np.nanmedian(values[~wt_mask])
        )
        diffs.append(diff)

    diffs = np.asarray(diffs, dtype=float)

    return {
        "median_perm_stat_WT_minus_NLGF": obs,
        "median_perm_p_two_sided": float(np.mean(np.abs(diffs) >= abs_obs - 1e-15)),
        "median_perm_n_permutations": int(len(diffs)),
        "median_perm_null_mean": float(np.mean(diffs)),
        "median_perm_null_sd": float(np.std(diffs, ddof=1)),
        "median_perm_null_abs_ge_obs": int(np.sum(np.abs(diffs) >= abs_obs - 1e-15)),
        "median_perm_ci2p5": float(np.quantile(diffs, 0.025)),
        "median_perm_ci97p5": float(np.quantile(diffs, 0.975)),
    }


rows = []

for _, row in discordant.iterrows():
    env = row["environment"]
    metric_col = row["original_metric"]

    sub = lfp.loc[
        lfp["environment"].astype(str).eq(str(env)),
        ["mouse_name", "Genotype", metric_col],
    ].dropna()

    res = exact_median_permutation(
        sub[metric_col].to_numpy(),
        sub["Genotype"].to_numpy(),
    )

    if res is None:
        continue

    median_sig = res["median_perm_p_two_sided"] < ALPHA

    if median_sig == bool(row["glmm_sig"]) and median_sig != bool(row["secondary_sig"]):
        closer = "GLMM-TMB"
    elif median_sig == bool(row["secondary_sig"]) and median_sig != bool(row["glmm_sig"]):
        closer = "secondary_test"
    elif median_sig == bool(row["glmm_sig"]) == bool(row["secondary_sig"]):
        closer = "both"
    else:
        closer = "neither"

    rows.append({
        "metric_group": row["metric_group"],
        "gamma_band": row["gamma_band"],
        "original_metric": metric_col,
        "metric": row["metric"],
        "environment": env,
        "discordance": row["concordance"],
        "glmm_family": row["glmm_family"],
        "glmm_contrast": row["glmm_contrast"],
        "glmm_estimate": row["glmm_estimate"],
        "glmm_p": row["glmm_p"],
        "glmm_sig": bool(row["glmm_sig"]),
        "secondary_test": row["secondary_test"],
        "secondary_p": row["secondary_p"],
        "secondary_sig": bool(row["secondary_sig"]),
        "n_WT": int(row["n_WT"]),
        "n_NLGF": int(row["n_NLGF"]),
        "WT_value": row["WT_value"],
        "NLGF_value": row["NLGF_value"],
        **res,
        "median_perm_sig": median_sig,
        "tiebreaker_closer_to_median_truth": closer,
    })


out = (
    pd.DataFrame(rows)
    .sort_values(["environment", "metric_group", "gamma_band"])
    .reset_index(drop=True)
)

out.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(
    out[
        [
            "environment",
            "metric",
            "gamma_band",
            "discordance",
            "glmm_p",
            "secondary_p",
            "median_perm_p_two_sided",
            "median_perm_sig",
            "tiebreaker_closer_to_median_truth",
        ]
    ].to_string(index=False)
)

Saved: /Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis/median_permutation_tiebreaker_discordant_glmmTMB_vs_secondary_test.csv
environment                            metric gamma_band  discordance   glmm_p  secondary_p  median_perm_p_two_sided  median_perm_sig tiebreaker_closer_to_median_truth
 open_field Theta–Fast gamma modulation index       fast glmmTMB_only 0.014661     0.135346                 0.327026            False                    secondary_test


In [79]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd

ALPHA = 0.05

base = Path(
    "/Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/"
    "DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis"
)

data_path = base / "concatenated_lfp_stats.csv"
concordance_path = base / "glmm-tmb_vs_secondary_test_concordance_FDR.csv"
out_path = base / "median_permutation_tiebreaker_discordant_glmmTMB_vs_secondary_test_FDR.csv"

lfp = pd.read_csv(data_path)
comp = pd.read_csv(concordance_path)

discordant = comp[comp["concordance_fdr"].isin(["GLMM-TMB_only_fdr", "secondary_only_fdr"])].copy()


def exact_median_permutation(values, labels, wt_label="WT", ko_label="NLGF"):
    values = np.asarray(values, dtype=float)
    labels = np.asarray(labels).astype(str)

    keep = np.isfinite(values) & np.isin(labels, [wt_label, ko_label])
    values = values[keep]
    labels = labels[keep]

    n_total = len(values)
    n_wt = int(np.sum(labels == wt_label))
    n_ko = int(np.sum(labels == ko_label))

    if n_wt == 0 or n_ko == 0:
        return None

    obs = float(
        np.nanmedian(values[labels == wt_label])
        - np.nanmedian(values[labels == ko_label])
    )
    abs_obs = abs(obs)

    diffs = []
    for wt_idx in combinations(range(n_total), n_wt):
        wt_mask = np.zeros(n_total, dtype=bool)
        wt_mask[list(wt_idx)] = True

        diff = float(
            np.nanmedian(values[wt_mask])
            - np.nanmedian(values[~wt_mask])
        )
        diffs.append(diff)

    diffs = np.asarray(diffs, dtype=float)

    return {
        "median_perm_stat_WT_minus_NLGF": obs,
        "median_perm_p_two_sided": float(np.mean(np.abs(diffs) >= abs_obs - 1e-15)),
        "median_perm_n_permutations": int(len(diffs)),
        "median_perm_null_mean": float(np.mean(diffs)),
        "median_perm_null_sd": float(np.std(diffs, ddof=1)),
        "median_perm_null_abs_ge_obs": int(np.sum(np.abs(diffs) >= abs_obs - 1e-15)),
        "median_perm_ci2p5": float(np.quantile(diffs, 0.025)),
        "median_perm_ci97p5": float(np.quantile(diffs, 0.975)),
    }


rows = []

for _, row in discordant.iterrows():
    env = row["environment"]
    metric_col = row["original_metric"]

    sub = lfp.loc[
        lfp["environment"].astype(str).eq(str(env)),
        ["mouse_name", "Genotype", metric_col],
    ].dropna()

    res = exact_median_permutation(
        sub[metric_col].to_numpy(),
        sub["Genotype"].to_numpy(),
    )

    if res is None:
        continue

    median_sig = res["median_perm_p_two_sided"] < ALPHA

    if median_sig == bool(row["glmm_sig"]) and median_sig != bool(row["secondary_sig"]):
        closer = "GLMM-TMB"
    elif median_sig == bool(row["secondary_sig"]) and median_sig != bool(row["glmm_sig"]):
        closer = "secondary_test"
    elif median_sig == bool(row["glmm_sig"]) == bool(row["secondary_sig"]):
        closer = "both"
    else:
        closer = "neither"

    rows.append({
        "metric_group": row["metric_group"],
        "gamma_band": row["gamma_band"],
        "original_metric": metric_col,
        "metric": row["metric"],
        "environment": env,
        "discordance": row["concordance_fdr"],
        "glmm_family": row["glmm_family"],
        "glmm_contrast": row["glmm_contrast"],
        "glmm_estimate": row["glmm_estimate"],
        "glmm_p": row["glmm_p"],
        "glmm_sig": bool(row["glmm_sig"]),
        "secondary_test": row["secondary_test"],
        "secondary_p": row["secondary_p"],
        "secondary_sig": bool(row["secondary_sig"]),
        "n_WT": int(row["n_WT"]),
        "n_NLGF": int(row["n_NLGF"]),
        "WT_value": row["WT_value"],
        "NLGF_value": row["NLGF_value"],
        **res,
        "median_perm_sig": median_sig,
        "tiebreaker_closer_to_median_truth": closer,
    })


out = (
    pd.DataFrame(rows)
    .sort_values(["environment", "metric_group", "gamma_band"])
    .reset_index(drop=True)
)

out.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(
    out[
        [
            "environment",
            "metric",
            "gamma_band",
            "discordance",
            "glmm_p",
            "secondary_p",
            "median_perm_p_two_sided",
            "median_perm_sig",
            "tiebreaker_closer_to_median_truth",
        ]
    ].to_string(index=False)
)

Saved: /Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis/median_permutation_tiebreaker_discordant_glmmTMB_vs_secondary_test_FDR.csv
 environment                            metric gamma_band        discordance   glmm_p  secondary_p  median_perm_p_two_sided  median_perm_sig tiebreaker_closer_to_median_truth
linear_track Theta–Fast gamma modulation index       fast  GLMM-TMB_only_fdr 0.001924     0.036042                 0.024887             True                              both
  open_field                          PAC slow       slow secondary_only_fdr 0.042852     0.001563                 0.004525             True                              both
  open_field Theta–Fast gamma modulation index       fast  GLMM-TMB_only_fdr 0.014661     0.135346                 0.327026            False                    secondary_test
  open_field Theta–Slow gamma modulation index       slow secondary_only_fdr 0.038527

In [80]:
# -- Significant GLMM-TMB / Step 3 secondary-test results summary -------------------
from pathlib import Path
from IPython.display import display, HTML
import pandas as pd

ALPHA = 0.05
summary_path = Path(OUTPUT_DIR) / "glmmTMB_vs_secondary_test_significant_results_by_gamma_band.csv"
full_path = Path(OUTPUT_DIR) / "glmmTMB_vs_secondary_test_concordance_by_gamma_band.csv"

if "glmmTMB_vs_secondary_test" in globals():
    sig_summary = glmmTMB_vs_secondary_test.loc[
        glmmTMB_vs_secondary_test["glmm_sig"] | glmmTMB_vs_secondary_test["secondary_sig"]
    ].copy()
elif summary_path.exists():
    sig_summary = pd.read_csv(summary_path)
elif full_path.exists():
    full = pd.read_csv(full_path)
    sig_summary = full.loc[full["glmm_sig"] | full["secondary_sig"]].copy()
else:
    raise FileNotFoundError(
        "Run the GLMM-TMB vs secondary-test concordance cell first."
    )

ordered_cols = [
    "metric", "environment", "gamma_band", "glmm_family",
    "glmm_estimate", "glmm_p", "glmm_sig_label",
    "secondary_test", "secondary_p", "secondary_sig_label", "secondary_statistic",
    "WT_value", "NLGF_value", "n_WT", "n_NLGF", "concordance",
]
ordered_cols = [col for col in ordered_cols if col in sig_summary.columns]

sig_summary = sig_summary.sort_values(["environment", "metric", "gamma_band"]).reset_index(drop=True)
sig_summary.to_csv(summary_path, index=False)

print(f"Significant GLMM-TMB or secondary-test rows: {len(sig_summary)}")
print(f"Saved: {summary_path}")

if sig_summary.empty:
    print(f"No significant GLMM-TMB or secondary-test genotype contrasts at alpha = {ALPHA}.")
else:
    display(HTML("<h3>Significant GLMM-TMB or secondary-test genotype effects</h3>"))
    display(
        sig_summary[ordered_cols].style
        .format({
            "glmm_estimate": "{:.6f}",
            "glmm_p": "{:.4f}",
            "secondary_p": "{:.4f}",
            "secondary_statistic": "{:.2f}",
            "WT_value": "{:.6f}",
            "NLGF_value": "{:.6f}",
        })
        .set_properties(**{"font-size": "11px"})
    )


Significant GLMM-TMB or secondary-test rows: 6
Saved: /Users/loukia/Library/CloudStorage/Dropbox-UCL/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/Analysis/LFP_analysis/glmmTMB_vs_secondary_test_significant_results_by_gamma_band.csv


,metric,environment,gamma_band,glmm_family,glmm_estimate,glmm_p,glmm_sig_label,secondary_test,secondary_p,secondary_sig_label,secondary_statistic,WT_value,NLGF_value,n_WT,n_NLGF,concordance
0,PAC slow,linear_track,slow,gamma,1.293371,0.0018,**,Welch's t-test,0.0116,*,3.27,0.000675,0.000189,8,9,both_significant
1,Theta–Fast gamma modulation index,linear_track,fast,beta,0.691334,0.0019,**,Student's t-test,0.0360,*,2.30,0.064006,0.033314,8,9,both_significant
2,Theta–Slow gamma modulation index,linear_track,slow,beta,0.664538,0.0032,**,Mann-Whitney U,0.0010,***,68.00,0.060694,0.024642,8,9,both_significant
3,PAC slow,open_field,slow,gamma,0.829007,0.0429,*,Mann-Whitney U,0.0016,**,67.00,0.000703,0.000363,8,9,both_significant
4,Theta–Fast gamma modulation index,open_field,fast,beta,0.618303,0.0147,*,Student's t-test,0.1353,ns,1.58,0.068590,0.040878,8,9,glmmTMB_only
5,Theta–Slow gamma modulation index,open_field,slow,beta,0.512533,0.0385,*,Student's t-test,0.0003,***,4.67,0.058463,0.034993,8,9,both_significant
